# CFTC TFF Nasdaq Mini 两周原件对年度文件
仅对 2011-06-21 与 2025-09-30 官方周报的合约209742三字段做原字节比对；不读市场收益或拟合。

In [1]:
import csv,hashlib,io,json,re,shlex,subprocess,tarfile,tempfile,zipfile
from pathlib import Path
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd().resolve();card=root/'research/stage1_cftc_nq_tff_weekly_parity_20261023';plan=json.loads((card/'plan.json').read_text());receipt=json.loads((card/'capture_receipts.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and receipt['plan_sha256']==sha(card/'plan.json') and receipt['GETs_attempted']==len(plan['cases'])==2
checks=[];paths=[card/'plan.json',card/'capture_receipts.json']
for case,row in zip(plan['cases'],receipt['captures'],strict=True):
 assert row['report_date']==case['report_date'] and row['requested_url']==case['official_weekly_url'] and row['curl_exit']==0 and row['http_status']=='200'
 for kind in ('body','headers'):
  entry=row[kind];path=root/entry['path'];assert path.stat().st_size==entry['bytes'] and sha(path)==entry['sha256'];paths.append(path)
 raw=(root/row['body']['path']).read_text(errors='replace');lines=raw.splitlines();matches=[i for i,line in enumerate(lines) if 'CFTC Code #209742' in line];assert len(matches)==1
 i=matches[0];assert 'NASDAQ' in lines[i-1] and 'CHICAGO MERCANTILE' in lines[i-1] and lines[i+1].strip()=='Positions'
 oi_match=re.search(r'Open Interest is\s+([\d,]+)',lines[i]);assert oi_match
 nums=[int(x.replace(',','')) for x in lines[i+2].split()];assert len(nums)==14
 weekly={'open_interest':int(oi_match.group(1).replace(',','')),'leveraged_long':nums[6],'leveraged_short':nums[7]}
 annual_path=root/case['annual_zip'];assert sha(annual_path)==case['annual_zip_sha256']
 with zipfile.ZipFile(annual_path) as source:
  names=[x for x in source.namelist() if x.lower().endswith('.txt')];assert len(names)==1
  rows=list(csv.DictReader(io.TextIOWrapper(source.open(names[0]),encoding='latin1')))
 candidates=[x for x in rows if x.get('CFTC_Contract_Market_Code','').strip()=='209742' and any(x.get(k,'').strip()==case['report_date'] for k in ('Report_Date_as_MM_DD_YYYY','Report_Date_as_YYYY-MM-DD'))]
 assert len(candidates)==1;annual_row=candidates[0]
 annual={'open_interest':int(annual_row['Open_Interest_All']),'leveraged_long':int(annual_row['Lev_Money_Positions_Long_All']),'leveraged_short':int(annual_row['Lev_Money_Positions_Short_All'])}
 checks.append({'report_date':case['report_date'],'official_special_release_date':case['official_special_release_date'],'weekly_url':row['effective_url'],'weekly_body_sha256':row['body']['sha256'],'annual_zip_sha256':case['annual_zip_sha256'],'weekly':weekly,'annual':annual,'exact_three_field_match':weekly==annual})
assert all(x['exact_three_field_match'] for x in checks)
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='cftc-weekly-parity-') as directory:
 archive=Path(directory)/'raw.tar.gz'
 with tarfile.open(archive,'w:gz') as output:
  for path in paths:output.add(path,arcname=str(path.relative_to(root)),recursive=False)
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote_path)).split()[0]==digest
 restored=Path(directory)/'restored.tar.gz'
 with restored.open('wb') as out:transfer=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote_path)],stdout=out,stderr=subprocess.PIPE,timeout=60)
 assert transfer.returncode==0 and sha(restored)==digest
 with tarfile.open(restored,'r:gz') as source:
  members={m.name:m for m in source.getmembers()};assert set(members)=={str(p.relative_to(root)) for p in paths}
  for path in paths:assert source.extractfile(members[str(path.relative_to(root))]).read()==path.read_bytes()
 evidence['nas_path']=remote_path
result={'status':'two_official_weekly_annual_NQ209742_three_field_parity_passed','plan_sha256':sha(card/'plan.json'),'GETs_consumed':2,'checks':checks,'NAS_evidence':evidence,'NAS_remote_and_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'historical_release_calendar_qualified':False,'all_history_first_public_vintage_qualified':False}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
summary={'experiment':card.name,'decision':'two_anomalous_weekly_annual_rows_match_timing_still_pending','objective_achieved':False,'result':'result.json','plan_sha256':result['plan_sha256'],'GETs_consumed':2,'weekly_cases_matched':2,'NAS_archive_sha256':evidence['sha256'],'NAS_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(summary['decision'],checks,evidence['sha256'])

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


two_anomalous_weekly_annual_rows_match_timing_still_pending [{'report_date': '2011-06-21', 'official_special_release_date': '2011-06-24', 'weekly_url': 'https://www.cftc.gov/sites/default/files/files/dea/cotarchives/2011/futures/financial_lf062111.htm', 'weekly_body_sha256': 'f22034d7c6b1963fd6c9ca354f2d02699a0cad4bdce857d202dcfc5fe435412b', 'annual_zip_sha256': '583a93474847f1070b213e2b12cfa73c48d9ba05f24f7d001a338de125468b57', 'weekly': {'open_interest': 270658, 'leveraged_long': 64032, 'leveraged_short': 77116}, 'annual': {'open_interest': 270658, 'leveraged_long': 64032, 'leveraged_short': 77116}, 'exact_three_field_match': True}, {'report_date': '2025-09-30', 'official_special_release_date': '2025-11-19', 'weekly_url': 'https://www.cftc.gov/sites/default/files/files/dea/cotarchives/2025/futures/financial_lf093025.htm', 'weekly_body_sha256': 'd97105b1ab2dfe958af0b52c524bf0e98ab625e41c6a882dc697d00c44fb3fdc', 'annual_zip_sha256': '2ea0cda6395f7dd6501c27422be3763e1f2f5b41b768cfd36b87

## 中文结论
# CFTC Nasdaq Mini TFF 两期周报与年度文件核对

冻结 CFTC 合约 `209742` 的两期官方逐周 TFF Futures Only 原文，各请求一次；确切响应字节、响应头、UTC 请求时刻和 SHA-256 已保留在本机与 Yanxuan NAS，远端哈希及逐文件限定恢复通过。响应头含 Cloudflare 临时 Cookie，原字节不提交 Git；Git 收据只保留哈希。

2011-06-21 报告的周报与年度 ZIP 同为：未平仓量 **270,658**、杠杆资金多头 **64,032**、空头 **77,116**。2025-09-30 报告同为 **278,188 / 61,911 / 63,535**。CFTC [特殊公告](https://www.cftc.gov/MarketReports/CommitmentsofTraders/HistoricalSpecialAnnouncements/index.htm)记载 2011-06-21 合约在 2011-06-24 发布时已调整未平仓量，并把 2025-09-30 报告的补发日期列为 2025-11-19；[历史索引](https://www.cftc.gov/MarketReports/CommitmentsofTraders/HistoricalViewable/index.htm)明确页面日期是**报告日**。

结论：`two_anomalous_weekly_annual_rows_match_timing_still_pending`。两期数值一致增强特定样本来源核对，不能扩展为全部历史周次的首发版本、完整实际发布日期或 09:46 决策前可得证明。CFTC [常见问题](https://www.cftc.gov/MarketReports/CommitmentsofTraders/index.htm)说明没有完整历史发布日期列表；后续须先建立保守发布时间与异常周次的未知处理，才可构造 Stage1 特征。0 市场结果读取、0 拟合、0 账户，SOXX 目标未达成。
